# Open-Ended Language Probabilities Across Layer Windows

This notebook reproduces the two open-ended story figures used in the paper:

1. prompt-averaged language probabilities across layer windows for PUD21+UD6;
2. the titleless micro-averaged comparison of controlled translation and open-ended INCLUDE.

It loads existing `lang_probs_<revision>.pt` artifacts and cached aggregation tables; it does not run model inference.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import torch

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "vis.py").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from analysis import (
    DOMAIN_CATEGORY_LABELS,
    OPENENDED_METHOD_ORDER_WITH_TUNED,
    MULTILINGUAL_PERFORMANCE_ASCENDING_MODEL_ORDER,
    diagnose_story_distribution_sums,
    discover_eval_runs,
    load_lang_probs_artifact,
    load_openended_run_manifest,
    micro_average_langdist_categories,
    micro_average_story_row_values,
    synthetic_model_display_name,
    translation_target_from_data_source,
)
from vis import (
    overlay_story_category_max_markers,
    plot_story_category_bar_grid,
    save_matplotlib_figure_bundle,
    set_matplotlib_paper_font,
)

LOG_ROOT = REPO_ROOT / "logs" / "evals"
CACHE_DIR = REPO_ROOT / ".analysis_cache" / "openended_language_probs_layer_windows"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
PAPER_MAIN_FIG_DIR = REPO_ROOT / "figs" / "paper" / "main"
PAPER_APPENDIX_FIG_DIR = REPO_ROOT / "figs" / "paper" / "appendix"

OPENENDED_METHODS = [
    method for method in OPENENDED_METHOD_ORDER_WITH_TUNED if method != "repr"
] + ["repr"]
PAPER_PUD_DATA_SOURCE = "pud21_ud6"
INCLUDE_DATA_SOURCE = "include_10lang_3domain_cap30"
OPENENDED_DATA_SOURCES = [PAPER_PUD_DATA_SOURCE, INCLUDE_DATA_SOURCE]
OPENENDED_MODEL_ORDER = list(MULTILINGUAL_PERFORMANCE_ASCENDING_MODEL_ORDER)
EXCLUDE_ENGLISH_PROMPTS = True

OPENENDED_CATEGORY_CACHE_PATH = CACHE_DIR / "openended_langdist_category_by_layer.parquet"

TRANSLATION_CATEGORY_CACHE_PATH = CACHE_DIR / "translation_langdist_category_by_layer.parquet"
TRANSLATION_SOURCE_CACHE_PATH = REPO_ROOT / ".analysis_cache" / "controlled_language_probs_layer_windows" / "translation_langdist_category_by_layer.parquet"

USE_CACHE = True
FORCE_REBUILD_OPENENDED_CACHE = False
FORCE_REBUILD_TRANSLATION_CACHE = False

BAR_ORDER = ["task_relevant", "english", "other"]
BAR_LABELS = {
    "task_relevant": "Avg. P(L=task-relevant)",
    "english": "P(L=English)",
    "other": "Avg. P(L=other)",
}

set_matplotlib_paper_font()


## Open-Ended Run Manifest

Select existing PUD/INCLUDE runs by config-derived metadata. Missing artifacts remain visible in the coverage table but are skipped during cache building.


In [ ]:
manifest = load_openended_run_manifest(
    LOG_ROOT,
    data_sources=OPENENDED_DATA_SOURCES,
    methods=OPENENDED_METHODS,
    require_prompt_metrics=True,
)

if manifest.empty:
    print("No matching open-ended runs found.")
else:
    manifest = manifest.copy()
    manifest["lang_probs_artifact_path"] = manifest.apply(
        lambda row: Path(row["exp_path"]) / f"lang_probs_{row['revision']}.pt",
        axis=1,
    )
    manifest["lang_probs_artifact_exists"] = manifest["lang_probs_artifact_path"].map(lambda path: Path(path).exists())
    display_cols = ["data_source", "method_label", "display_model_name", "exp_id", "lang_probs_artifact_exists"]
    print(f"Selected open-ended runs: {len(manifest):,}")
    display(manifest[display_cols].sort_values(["data_source", "method_label", "display_model_name"]))

    print("Coverage by dataset / estimator:")
    display(
        manifest.pivot_table(
            index="data_source",
            columns="method_label",
            values="display_model_name",
            aggfunc="nunique",
            fill_value=0,
        ).reindex(index=OPENENDED_DATA_SOURCES, columns=OPENENDED_METHODS)
    )


## Category/Layer Cache Builders

In [ ]:
def _openended_dataset_label(data_source):
    if data_source == "include_10lang_3domain_cap30":
        return "INCLUDE cap30"
    return DOMAIN_CATEGORY_LABELS.get(data_source, str(data_source))


def _artifact_method_for_label(method_label):
    return "repr_gmm" if method_label == "repr" else "decoding"


def _compute_openended_langdist_category_arrays(probs, langs, prompt_langs):
    lang_to_idx = {lang: idx for idx, lang in enumerate(langs)}
    en_idx = lang_to_idx.get("en")
    if en_idx is None:
        english = np.full(probs.shape[:2], np.nan, dtype=np.float32)
        english_counts = np.zeros(probs.shape[0], dtype=np.int32)
    else:
        english = probs[:, :, en_idx]
        english_counts = np.ones(probs.shape[0], dtype=np.int32)

    task_relevant = np.full(probs.shape[:2], np.nan, dtype=np.float32)
    other = np.full(probs.shape[:2], np.nan, dtype=np.float32)
    other_max = np.full(probs.shape[:2], np.nan, dtype=np.float32)
    other_sum = np.full(probs.shape[:2], np.nan, dtype=np.float32)
    other_sq_sum = np.full(probs.shape[:2], np.nan, dtype=np.float32)
    other_counts = np.zeros(probs.shape[0], dtype=np.int32)

    for prompt_idx, prompt_lang in enumerate(prompt_langs):
        prompt_lang = str(prompt_lang)
        rel_idx = lang_to_idx.get(prompt_lang)
        if rel_idx is None:
            continue
        task_relevant[prompt_idx, :] = probs[prompt_idx, :, rel_idx]
        other_indices = [
            idx for lang, idx in lang_to_idx.items()
            if lang not in {"en", prompt_lang}
        ]
        other_counts[prompt_idx] = len(other_indices)
        if other_indices:
            other_values = probs[prompt_idx][:, other_indices]
            other[prompt_idx, :] = other_values.mean(axis=-1)
            other_max[prompt_idx, :] = other_values.max(axis=-1)
            other_sum[prompt_idx, :] = other_values.sum(axis=-1)
            other_sq_sum[prompt_idx, :] = np.square(other_values).sum(axis=-1)

    return {
        "task_relevant": (
            task_relevant,
            task_relevant,
            np.ones(probs.shape[0], dtype=np.int32),
            task_relevant,
            np.square(task_relevant),
        ),
        "english": (english, english, english_counts, english, np.square(english)),
        "other": (other, other_max, other_counts, other_sum, other_sq_sum),
    }


def build_openended_langdist_category_rows(manifest):
    """Expand open-ended LangDists into prompt-layer category rows."""
    columns = [
        "dataset",
        "data_source",
        "dataset_label",
        "method_label",
        "display_model_name",
        "model_name",
        "exp_id",
        "prompt_id",
        "prompt_lang",
        "layer",
        "layer_norm",
        "prob_category",
        "category_lang_count",
        "category_prob_sum",
        "category_prob_sq_sum",
        "prob_langdist",
        "prob_langdist_max",
    ]
    if manifest.empty:
        return pd.DataFrame(columns=columns)

    frames = []
    available = manifest[manifest["lang_probs_artifact_exists"]].copy()
    for row in tqdm(available.itertuples(index=False), total=len(available), desc="Loading open-ended lang_probs", unit="run"):
        artifact = load_lang_probs_artifact(row.lang_probs_artifact_path)
        method_key = _artifact_method_for_label(row.method_label)
        payload = artifact.get("methods", {}).get(method_key)
        if payload is None:
            print(f"Missing method payload {method_key!r} for {row.exp_id}")
            continue

        langs = list(payload["langs"])
        probs = payload["probs"].to(dtype=torch.float32).numpy()
        prompt_ids = list(artifact.get("prompt_ids") or [f"prompt-{i}" for i in range(probs.shape[0])])
        prompt_langs = list(artifact.get("prompt_langs") or [None] * probs.shape[0])
        layer_indices = list(artifact.get("layer_indices") or list(range(probs.shape[1])))
        max_layer = max(layer_indices) if layer_indices else max(1, probs.shape[1] - 1)
        max_layer = max(max_layer, 1)

        prompt_mask = np.ones(probs.shape[0], dtype=bool)
        if EXCLUDE_ENGLISH_PROMPTS:
            prompt_mask &= np.array([lang != "en" for lang in prompt_langs], dtype=bool)
        if not prompt_mask.any():
            continue

        probs = probs[prompt_mask]
        prompt_ids = [pid for pid, keep in zip(prompt_ids, prompt_mask) if keep]
        prompt_langs = [lang for lang, keep in zip(prompt_langs, prompt_mask) if keep]
        category_arrays = _compute_openended_langdist_category_arrays(probs, langs, prompt_langs)

        base = pd.MultiIndex.from_product(
            [range(len(prompt_ids)), range(len(layer_indices))],
            names=["prompt_offset", "layer_offset"],
        ).to_frame(index=False)
        base["prompt_id"] = base["prompt_offset"].map(lambda idx: prompt_ids[int(idx)])
        base["prompt_lang"] = base["prompt_offset"].map(lambda idx: prompt_langs[int(idx)])
        base["layer"] = base["layer_offset"].map(lambda idx: int(layer_indices[int(idx)]))
        base["layer_norm"] = base["layer"] / float(max_layer)

        for category, (values, max_values, counts, value_sums, value_sq_sums) in category_arrays.items():
            frame = base.copy()
            frame["prob_category"] = category
            frame["category_lang_count"] = frame["prompt_offset"].map(lambda idx: int(counts[int(idx)]))
            frame["category_prob_sum"] = value_sums.reshape(-1)
            frame["category_prob_sq_sum"] = value_sq_sums.reshape(-1)
            frame["prob_langdist"] = values.reshape(-1)
            frame["prob_langdist_max"] = max_values.reshape(-1)
            frame = frame.drop(columns=["prompt_offset", "layer_offset"])
            frame = frame[frame["prob_langdist"].notna()].copy()
            frame["dataset"] = row.data_source
            frame["data_source"] = row.data_source
            frame["dataset_label"] = _openended_dataset_label(row.data_source)
            frame["method_label"] = row.method_label
            frame["display_model_name"] = row.display_model_name
            frame["model_name"] = row.model_name
            frame["exp_id"] = row.exp_id
            frames.append(frame[columns])

    if not frames:
        return pd.DataFrame(columns=columns)
    return pd.concat(frames, ignore_index=True)


def load_or_build_openended_category_cache():
    """Load the open-ended category cache or rebuild it from artifacts."""
    if USE_CACHE and OPENENDED_CATEGORY_CACHE_PATH.exists() and not FORCE_REBUILD_OPENENDED_CACHE:
        cached = pd.read_parquet(OPENENDED_CATEGORY_CACHE_PATH)
        required_cache_columns = {"prob_langdist_max", "category_prob_sq_sum"}
        if required_cache_columns.issubset(cached.columns):
            print(f"Loading open-ended category/layer cache: {OPENENDED_CATEGORY_CACHE_PATH.relative_to(REPO_ROOT)}", flush=True)
            return cached
        print(f"Rebuilding open-ended cache without exact micro-averaging statistics: {OPENENDED_CATEGORY_CACHE_PATH.relative_to(REPO_ROOT)}", flush=True)
    df = build_openended_langdist_category_rows(manifest)
    df.to_parquet(OPENENDED_CATEGORY_CACHE_PATH, index=False)
    df.to_csv(OPENENDED_CATEGORY_CACHE_PATH.with_suffix(".csv"), index=False)
    print(f"Wrote open-ended category/layer cache: {OPENENDED_CATEGORY_CACHE_PATH.relative_to(REPO_ROOT)}")
    return df


def load_or_copy_translation_category_cache():
    if USE_CACHE and TRANSLATION_CATEGORY_CACHE_PATH.exists() and not FORCE_REBUILD_TRANSLATION_CACHE:
        print(f"Loading translation category/layer cache: {TRANSLATION_CATEGORY_CACHE_PATH.relative_to(REPO_ROOT)}", flush=True)
        return pd.read_parquet(TRANSLATION_CATEGORY_CACHE_PATH)
    if not TRANSLATION_SOURCE_CACHE_PATH.exists():
        print(f"Translation cache not found: {TRANSLATION_SOURCE_CACHE_PATH.relative_to(REPO_ROOT)}")
        return pd.DataFrame()
    print(f"Loading controlled translation cache: {TRANSLATION_SOURCE_CACHE_PATH.relative_to(REPO_ROOT)}", flush=True)
    df = pd.read_parquet(TRANSLATION_SOURCE_CACHE_PATH)
    if "method_label" not in df.columns:
        df = df.copy()
        df["method_label"] = "target-start"
    df.to_parquet(TRANSLATION_CATEGORY_CACHE_PATH, index=False)
    df.to_csv(TRANSLATION_CATEGORY_CACHE_PATH.with_suffix(".csv"), index=False)
    print(f"Wrote translation category/layer cache: {TRANSLATION_CATEGORY_CACHE_PATH.relative_to(REPO_ROOT)}")
    return df


## Load Caches

In [ ]:
category_per_layer_df = load_or_build_openended_category_cache()
category_per_layer_df = category_per_layer_df[
    category_per_layer_df["data_source"].isin(OPENENDED_DATA_SOURCES)
].copy()
print(f"Open-ended category/layer rows: {len(category_per_layer_df):,}")

translation_category_per_layer_df = load_or_copy_translation_category_cache()
print(f"Translation category/layer rows: {len(translation_category_per_layer_df):,}")


## Paper Figure: PUD21+UD6 Across Layer Windows

Rows are LLID estimators, columns are layer windows, and representation is intentionally last.


In [ ]:
OPENENDED_LAYER_WINDOWS = [
    {
        "slug": "first50",
        "label": "First 50% Layers",
        "layer_min": 0.00,
        "layer_max": 0.50,
        "include_min": True,
        "include_max": False,
    },
    {
        "slug": "middle50_75",
        "label": "50%-75% Layers",
        "layer_min": 0.50,
        "layer_max": 0.75,
        "include_min": True,
        "include_max": False,
    },
    {
        "slug": "final25",
        "label": "75%-100% Layers",
        "layer_min": 0.75,
        "layer_max": 1.00,
        "include_min": True,
        "include_max": True,
    },
]
def summarize_story_layer_windows(category_df, *, group_cols, windows, use_category_max=False):
    """Micro-average LangDist categories over the requested layer windows."""
    summaries = []
    for window in tqdm(windows, desc="Summarizing layer windows", unit="window"):
        common_kwargs = dict(
            group_cols=group_cols,
            layer_min=window["layer_min"],
            layer_max=window["layer_max"],
            include_min=window["include_min"],
            include_max=window["include_max"],
            keep_category_lang_count=True,
        )
        if use_category_max:
            summary = micro_average_story_row_values(
                category_df, prob_col="prob_langdist_max", **common_kwargs
            )
        else:
            summary = micro_average_langdist_categories(category_df, **common_kwargs)
        summary["layer_window"] = window["label"]
        summary["layer_window_slug"] = window["slug"]
        summaries.append(summary)
    return pd.concat(summaries, ignore_index=True) if summaries else pd.DataFrame()


def order_models_default(summary_df, *, base_order=OPENENDED_MODEL_ORDER):
    if summary_df.empty or "display_model_name" not in summary_df.columns:
        return list(base_order)
    present = set(summary_df["display_model_name"].dropna())
    ordered = [model for model in base_order if model in present]
    extras = sorted(model for model in present if model not in set(base_order))
    return ordered + extras


def print_model_order(label, models, summary_df):
    english = (
        summary_df[summary_df["prob_category"].eq("english")]
        .groupby("display_model_name", dropna=False)["mean_prob"]
        .mean()
    )
    print(label)
    for model in models:
        value = english.get(model, np.nan)
        print(f"  {model}: English mean={value:.4f}" if pd.notna(value) else f"  {model}: English mean=NA")


In [ ]:
pud_category_df = category_per_layer_df[
    category_per_layer_df["data_source"].eq(PAPER_PUD_DATA_SOURCE)
].copy()
pud_story_summary = summarize_story_layer_windows(
    pud_category_df,
    group_cols=("data_source", "dataset_label", "method_label", "display_model_name"),
    windows=OPENENDED_LAYER_WINDOWS,
)
pud_story_max_summary = summarize_story_layer_windows(
    pud_category_df,
    group_cols=("data_source", "dataset_label", "method_label", "display_model_name"),
    windows=OPENENDED_LAYER_WINDOWS,
    use_category_max=True,
)
label = _openended_dataset_label(PAPER_PUD_DATA_SOURCE)
models = order_models_default(pud_story_summary)
print_model_order(f"Model order for {label}", models, pud_story_summary)
print(f"Plotting {label}: {len(pud_story_summary):,} rows", flush=True)
fig = plot_story_category_bar_grid(
    pud_story_summary,
    row_col="method_label",
    row_order=OPENENDED_METHODS,
    col_col="layer_window",
    col_order=[window["label"] for window in OPENENDED_LAYER_WINDOWS],
    models=models,
    category_order=BAR_ORDER,
    category_labels=BAR_LABELS,
    probability_label="Avg. Lang Prob across Prompts and Layers",
    shared_ylabel="Avg. Lang Prob across Prompts and Layers",
    shared_ylabel_x=0.012,
    left=0.075,
    title=f"[{label}] Language Probabilities Across Layer Windows",
    subtitle=(
        "Rows = LLID estimator, columns = layer window. English prompt cases are excluded; "
        "bars pool prompt-layer-language probabilities within each category; "
        "diamonds average prompt-layer category maxima; error bars show SE for both."
    ),
    unavailable_label=f"open-ended {label}",
    figsize=(15.0, 2.8 * len(OPENENDED_METHODS)),
    legend_y=0.115,
    bottom=0.24,
    top=0.88,
    title_y=0.995,
    subtitle_y=0.960,
    subtitle_linespacing=0.9,
    sharey=True,
    row_label_fontweight="bold",
    row_label_fontsize=15,
    col_title_fontweight="bold",
    x_tick_label_pad=5.0,
)
if fig is not None:
    overlay_story_category_max_markers(
        fig,
        pud_story_summary,
        pud_story_max_summary,
        row_col="method_label",
        col_col="layer_window",
        row_order=OPENENDED_METHODS,
        col_order=[window["label"] for window in OPENENDED_LAYER_WINDOWS],
        models=models,
        category_order=BAR_ORDER,
        legend_y=0.115,
        legend_fontsize=18,
        x_edge_padding=0.46,
        x_tick_shift_points=14.0,
    )
    save_matplotlib_figure_bundle(
        fig,
        PAPER_APPENDIX_FIG_DIR / "09_pud_ud_layer_windows",
    )
    plt.show()


## Paper Figure: Controlled Translation and Open-Ended INCLUDE

In [ ]:
COMPARISON_WINDOWS = [
    window for window in OPENENDED_LAYER_WINDOWS
    if window["slug"] in {"middle50_75", "final25"}
]

COMPARISON_MODEL_LABEL_MAP = {
    "Llama-2-7B": "Llama2",
    "Llama-3.1-8B": "Llama3.1",
    "Llama-3.1-8B-Instruct": "Llama3.1-Instr",
    "Apertus-8B": "Apertus",
    "Apertus-8B-Instruct": "Apertus-Instr",
    "Mistral-Nemo-Instruct": "Nemo-Instr",
    "Aya-23-8B": "Aya-23",
    "EuroLLM-9B": "EuroLLM",
    "EuroLLM-9B-Instruct": "EuroLLM-Instr",
    "OLMo-2-1124-7B": "OLMo2",
}

COMPARISON_ROW_TEMP_EXCLUDE_MODELS = {"OLMo-2-1124-7B"}

COMPARISON_TASK_PANEL_ORDER = [
    "Controlled: translation\nStart(w)",
    "Open-ended: INCLUDE\nRaw Logitlens Top-p",
    "Open-ended: INCLUDE\nTunedLens Top-p",
    "Open-ended: INCLUDE\nRepr-GMM",
]
COMPARISON_TASK_PANEL_TITLE_MAP = {
    "Controlled: translation\nStart(w)": "Start(w)",
    "Open-ended: INCLUDE\nRaw Logitlens Top-p": "Raw Logitlens Top-p",
    "Open-ended: INCLUDE\nTunedLens Top-p": "TunedLens Top-p",
    "Open-ended: INCLUDE\nRepr-GMM": "Repr-GMM",
}
COMPARISON_ROW_ORDER = ["50%-75% Layers", "75%-100% Layers"]
COMPARISON_ROW_LABEL_MAP = {
    "50%-75% Layers": "Layers\n50%-75%",
    "75%-100% Layers": "Layers\n75%-100%",
}


COMPARISON_BAR_LABELS = dict(BAR_LABELS)
COMPARISON_BAR_LABELS["other"] = "P(L=other)"

def comparison_task_panel_label(method_label):
    return {
        "target-start": "Controlled: translation\nStart(w)",
        "raw-rtopp": "Open-ended: INCLUDE\nRaw Logitlens Top-p",
        "tuned-rtopp": "Open-ended: INCLUDE\nTunedLens Top-p",
        "repr": "Open-ended: INCLUDE\nRepr-GMM",
    }.get(method_label, method_label)


def add_late_window_comparison_labels(summary_df):
    """Add panel and layer-row labels used by the comparison figure."""
    df = summary_df.copy()
    if df.empty:
        return df
    df["comparison_task_panel"] = df["method_label"].map(comparison_task_panel_label)
    df["comparison_layer_row"] = df["layer_window"].map(COMPARISON_ROW_LABEL_MAP).fillna(df["layer_window"])
    return df


def _story_grid_ymax(df, panel_names):
    if df.empty:
        return 1.0
    panel_df = df[df["comparison_task_panel"].isin(panel_names)].copy()
    if panel_df.empty:
        return 1.0
    values = pd.to_numeric(panel_df["mean_prob"], errors="coerce")
    if "stderr_prob" in panel_df.columns:
        values = values + pd.to_numeric(panel_df["stderr_prob"], errors="coerce").fillna(0.0)
    elif "std_prob" in panel_df.columns:
        values = values + pd.to_numeric(panel_df["std_prob"], errors="coerce").fillna(0.0)
    values = values.replace([np.inf, -np.inf], np.nan).dropna()
    if values.empty:
        return 1.0
    return max(1e-6, min(1.0, float(values.max()) * 1.18))


def decorate_late_window_comparison_by_window_rows(fig, plot_df):
    n_rows = len(COMPARISON_ROW_ORDER)
    n_cols = len(COMPARISON_TASK_PANEL_ORDER)
    panel_axes = np.asarray(fig.axes[: n_rows * n_cols], dtype=object).reshape(n_rows, n_cols)

    fig.subplots_adjust(hspace=0.035, wspace=0.005)

    controlled_block_shift = -0.006
    for ax in panel_axes[:, :1].ravel():
        pos = ax.get_position()
        ax.set_position([pos.x0 + controlled_block_shift, pos.y0, pos.width, pos.height])

    # Lay out the INCLUDE block independently: a clearer dataset break, then
    # tighter gaps and wider axes within the open-ended group.
    controlled_right = panel_axes[0, 0].get_position().x1
    include_right = panel_axes[0, -1].get_position().x1
    include_left = controlled_right + 0.040
    include_inner_gap = 0.006
    include_panel_width = (include_right - include_left - 2 * include_inner_gap) / 3
    for row_idx in range(n_rows):
        for include_idx, col_idx in enumerate(range(1, n_cols)):
            pos = panel_axes[row_idx, col_idx].get_position()
            x0 = include_left + include_idx * (include_panel_width + include_inner_gap)
            panel_axes[row_idx, col_idx].set_position([x0, pos.y0, include_panel_width, pos.height])

    controlled_ylim = _story_grid_ymax(plot_df, [COMPARISON_TASK_PANEL_ORDER[0]])
    include_ylim = _story_grid_ymax(plot_df, COMPARISON_TASK_PANEL_ORDER[1:])
    for ax in panel_axes[:, 0]:
        ax.set_ylim(0.0, controlled_ylim)
    for ax in panel_axes[:, 1:].ravel():
        ax.set_ylim(0.0, include_ylim)
        ax.yaxis.set_major_locator(MaxNLocator(nbins=5, min_n_ticks=4))

    for col_idx, panel_name in enumerate(COMPARISON_TASK_PANEL_ORDER):
        panel_axes[0, col_idx].set_title(
            COMPARISON_TASK_PANEL_TITLE_MAP.get(panel_name, panel_name),
            fontsize=18,
            pad=9,
            fontweight="bold",
        )

    for ax in panel_axes.ravel():
        ax.tick_params(axis="both", labelsize=15)
        ax.xaxis.label.set_size(16)
        ax.yaxis.label.set_size(17)
    for ax in panel_axes[:, 0]:
        ax.yaxis.label.set_size(18)
        ax.yaxis.label.set_fontweight("bold")
        ax.yaxis.set_label_coords(-0.10, 0.5)
    for ax in panel_axes[:, 1]:
        ax.tick_params(axis="y", left=True, labelleft=True, labelsize=15)
        for tick_label in ax.get_yticklabels():
            tick_label.set_fontweight("bold")

    if fig.legends:
        for text in fig.legends[0].get_texts():
            text.set_fontsize(19)

    for ax in panel_axes[-1, :]:
        ax.tick_params(axis="x", labelsize=17)

    if getattr(fig, "_supylabel", None) is not None:
        fig._supylabel.set_fontsize(18)

    panel_top = max(ax.get_position().y1 for ax in panel_axes.ravel())
    dataset_label_y = panel_top + 0.062
    dataset_underline_y = dataset_label_y - 0.007
    dataset_spans = [
        (0, 0, "Controlled: translation"),
        (1, 3, "Open-ended: INCLUDE"),
    ]
    for start_idx, end_idx, label in dataset_spans:
        left = panel_axes[0, start_idx].get_position().x0
        right = panel_axes[0, end_idx].get_position().x1
        fig.text((left + right) / 2.0, dataset_label_y, label, ha="center", va="bottom", fontsize=18, fontweight="bold")
        width = right - left
        fig.add_artist(
            plt.Line2D(
                [left + 0.025 * width, right - 0.025 * width],
                [dataset_underline_y, dataset_underline_y],
                transform=fig.transFigure,
                color="#222222",
                linewidth=0.9,
                alpha=0.85,
            )
        )

    sep_x = (panel_axes[0, 0].get_position().x1 + panel_axes[0, 1].get_position().x0) / 2.0 - 0.010
    sep_y0 = min(ax.get_position().y0 for ax in panel_axes.ravel())
    sep_y1 = dataset_label_y + 0.024
    fig.add_artist(
        plt.Line2D(
            [sep_x, sep_x],
            [sep_y0, sep_y1],
            transform=fig.transFigure,
            color="#777777",
            linewidth=0.8,
            alpha=0.65,
        )
    )

TABLE1_REVERSE_MODEL_ORDER = list(MULTILINGUAL_PERFORMANCE_ASCENDING_MODEL_ORDER)

translation_df = translation_category_per_layer_df.copy()
if not translation_df.empty and "method_label" not in translation_df.columns:
    translation_df["method_label"] = "target-start"

COMBINED_LANGDIST_CACHE_PATH = CACHE_DIR / "combined_late_layer_langdist_by_language.parquet"


def _layer_offsets_for_window(layer_indices, window):
    max_layer = max(max(layer_indices) if layer_indices else 1, 1)
    layer_norm = np.asarray(layer_indices, dtype=float) / float(max_layer)
    mask = np.ones_like(layer_norm, dtype=bool)
    if window["layer_min"] is not None:
        mask &= layer_norm >= window["layer_min"] if window["include_min"] else layer_norm > window["layer_min"]
    if window["layer_max"] is not None:
        mask &= layer_norm <= window["layer_max"] if window["include_max"] else layer_norm < window["layer_max"]
    return np.flatnonzero(mask)


def _category_for_translation(lang, prompt_lang, requested_tgt_lang):
    if lang == "en":
        return "english"
    if lang in {prompt_lang, requested_tgt_lang}:
        return "task_relevant"
    return "other"


def _category_for_include(lang, prompt_lang):
    if lang == "en":
        return "english"
    if lang == prompt_lang:
        return "task_relevant"
    return "other"


def _accumulate_language_sums(*, probs, langs, prompt_langs, layer_indices, windows, base_fields, category_fn, skip_prompt_fn):
    records = []
    probs = probs.astype(np.float32, copy=False)
    for window in windows:
        layer_offsets = _layer_offsets_for_window(layer_indices, window)
        if layer_offsets.size == 0:
            continue
        for prompt_idx, prompt_lang in enumerate(prompt_langs):
            prompt_lang = None if prompt_lang is None else str(prompt_lang)
            if skip_prompt_fn(prompt_lang):
                continue
            for lang_idx, lang in enumerate(langs):
                category = category_fn(str(lang), prompt_lang)
                values = probs[prompt_idx, layer_offsets, lang_idx]
                finite = np.isfinite(values)
                if not finite.any():
                    continue
                records.append({
                    **base_fields,
                    "layer_window": window["label"],
                    "layer_window_slug": window["slug"],
                    "prob_category": category,
                    "lang": str(lang),
                    "sum_prob": float(values[finite].sum()),
                    "sum_sq_prob": float(np.square(values[finite]).sum()),
                    "n_values": int(finite.sum()),
                })
    return records


def _load_translation_runs_for_language_agg():
    if translation_df.empty:
        return pd.DataFrame()
    run_df = translation_df[["exp_id", "data_source", "display_model_name", "model_name", "requested_tgt_lang"]].drop_duplicates().copy()
    discovered = discover_eval_runs(LOG_ROOT, require_prompt_metrics=True)
    if discovered.empty:
        return pd.DataFrame()
    discovered = discovered[["exp_id", "exp_path", "revision"]].drop_duplicates("exp_id")
    return run_df.merge(discovered, on="exp_id", how="left")


def build_language_aggregates():
    """Aggregate per-language probability moments across selected runs."""
    if USE_CACHE and COMBINED_LANGDIST_CACHE_PATH.exists():
        cached = pd.read_parquet(COMBINED_LANGDIST_CACHE_PATH)
        if "sum_sq_prob" in cached.columns:
            print(f"Loading language aggregation cache: {COMBINED_LANGDIST_CACHE_PATH.relative_to(REPO_ROOT)}")
            return cached
        print(f"Rebuilding language aggregation cache without sum_sq_prob: {COMBINED_LANGDIST_CACHE_PATH.relative_to(REPO_ROOT)}")

    records = []
    translation_runs = _load_translation_runs_for_language_agg()
    for row in tqdm(translation_runs.itertuples(index=False), total=len(translation_runs), desc="Aggregating translation language rows", unit="run"):
        if not isinstance(row.exp_path, str):
            continue
        artifact_path = Path(row.exp_path) / f"lang_probs_{row.revision}.pt"
        if not artifact_path.exists():
            print(f"Missing translation lang_probs artifact: {artifact_path}")
            continue
        artifact = load_lang_probs_artifact(artifact_path)
        payload = artifact.get("methods", {}).get("decoding", {})
        if "probs" not in payload:
            continue
        langs = list(payload.get("langs") or [])
        probs = payload["probs"].to(dtype=torch.float32).numpy()
        prompt_langs = list(artifact.get("prompt_langs") or [None] * probs.shape[0])
        layer_indices = list(artifact.get("layer_indices") or list(range(probs.shape[1])))
        requested_tgt_lang = row.requested_tgt_lang
        records.extend(_accumulate_language_sums(
            probs=probs,
            langs=langs,
            prompt_langs=prompt_langs,
            layer_indices=layer_indices,
            windows=COMPARISON_WINDOWS,
            base_fields={"method_label": "target-start", "display_model_name": row.display_model_name},
            category_fn=lambda lang, prompt_lang, requested_tgt_lang=requested_tgt_lang: _category_for_translation(lang, prompt_lang, requested_tgt_lang),
            skip_prompt_fn=lambda prompt_lang, requested_tgt_lang=requested_tgt_lang: prompt_lang == "en" or requested_tgt_lang == "en",
        ))

    include_runs = manifest[
        manifest["data_source"].eq(INCLUDE_DATA_SOURCE)
        & manifest["method_label"].isin(["raw-rtopp", "tuned-rtopp", "repr"])
        & manifest["lang_probs_artifact_exists"]
    ].copy()
    for row in tqdm(include_runs.itertuples(index=False), total=len(include_runs), desc="Aggregating INCLUDE language rows", unit="run"):
        artifact = load_lang_probs_artifact(row.lang_probs_artifact_path)
        payload = artifact.get("methods", {}).get(_artifact_method_for_label(row.method_label), {})
        if "probs" not in payload:
            continue
        langs = list(payload.get("langs") or [])
        probs = payload["probs"].to(dtype=torch.float32).numpy()
        prompt_langs = list(artifact.get("prompt_langs") or [None] * probs.shape[0])
        layer_indices = list(artifact.get("layer_indices") or list(range(probs.shape[1])))
        records.extend(_accumulate_language_sums(
            probs=probs,
            langs=langs,
            prompt_langs=prompt_langs,
            layer_indices=layer_indices,
            windows=COMPARISON_WINDOWS,
            base_fields={"method_label": row.method_label, "display_model_name": row.display_model_name},
            category_fn=lambda lang, prompt_lang: _category_for_include(lang, prompt_lang),
            skip_prompt_fn=lambda prompt_lang: prompt_lang == "en",
        ))

    agg = pd.DataFrame.from_records(records)
    if not agg.empty:
        agg = agg.groupby(["method_label", "display_model_name", "layer_window", "layer_window_slug", "prob_category", "lang"], dropna=False).agg(
            sum_prob=("sum_prob", "sum"),
            n_values=("n_values", "sum"),
            sum_sq_prob=("sum_sq_prob", "sum"),
        ).reset_index()
        agg["lang_mean_prob"] = agg["sum_prob"] / agg["n_values"].clip(lower=1)
    agg.to_parquet(COMBINED_LANGDIST_CACHE_PATH, index=False)
    agg.to_csv(COMBINED_LANGDIST_CACHE_PATH.with_suffix(".csv"), index=False)
    print(f"Wrote language aggregation cache: {COMBINED_LANGDIST_CACHE_PATH.relative_to(REPO_ROOT)}")
    return agg


def summarize_language_aggregates(agg):
    """Convert language-level moments into micro-averaged category statistics."""
    if agg.empty:
        return pd.DataFrame(columns=["method_label", "display_model_name", "layer_window", "layer_window_slug", "prob_category", "mean_prob"])
    group_cols = ["method_label", "display_model_name", "layer_window", "layer_window_slug", "prob_category"]
    out = agg.groupby(group_cols, dropna=False).agg(
        sum_prob=("sum_prob", "sum"),
        n_values=("n_values", "sum"),
        sum_sq_prob=("sum_sq_prob", "sum"),
    ).reset_index()
    out["mean_prob"] = out["sum_prob"] / out["n_values"].clip(lower=1)
    numerator = (out["sum_sq_prob"] - (out["sum_prob"] ** 2) / out["n_values"].clip(lower=1)).clip(lower=0.0)
    out["var_prob"] = numerator / (out["n_values"] - 1).clip(lower=1)
    out.loc[out["n_values"] <= 1, "var_prob"] = 0.0
    out["std_prob"] = np.sqrt(out["var_prob"])
    out["stderr_prob"] = out["std_prob"] / np.sqrt(out["n_values"].clip(lower=1))
    return out

language_agg = build_language_aggregates()
print(f"language aggregation rows: {len(language_agg):,}")

micro_summary = summarize_language_aggregates(language_agg)
print(f"Micro-averaged comparison rows: {len(micro_summary):,}")


In [ ]:
# Aggregate the strongest other-language probability used by Figure 4.
MAX_OTHER_LANGDIST_CACHE_PATH = CACHE_DIR / "combined_late_layer_max_other_langdist.parquet"


def _accumulate_max_other_sums(
    *, probs, langs, prompt_langs, layer_indices, windows, base_fields, other_lang_fn, skip_prompt_fn
):
    records = []
    probs = probs.astype(np.float32, copy=False)
    for window in windows:
        layer_offsets = _layer_offsets_for_window(layer_indices, window)
        if layer_offsets.size == 0:
            continue
        for prompt_idx, prompt_lang in enumerate(prompt_langs):
            prompt_lang = None if prompt_lang is None else str(prompt_lang)
            if skip_prompt_fn(prompt_lang):
                continue
            other_indices = [
                lang_idx
                for lang_idx, lang in enumerate(langs)
                if other_lang_fn(str(lang), prompt_lang)
            ]
            if not other_indices:
                continue
            values = probs[prompt_idx, layer_offsets][:, other_indices]
            finite = np.isfinite(values)
            valid_rows = finite.any(axis=1)
            if not valid_rows.any():
                continue
            max_values = np.where(finite, values, -np.inf).max(axis=1)[valid_rows]
            records.append({
                **base_fields,
                "layer_window": window["label"],
                "layer_window_slug": window["slug"],
                "prob_category": "other",
                "lang": "__max_other__",
                "sum_prob": float(max_values.sum()),
                "sum_sq_prob": float(np.square(max_values).sum()),
                "n_values": int(max_values.size),
            })
    return records


def build_max_other_language_aggregates():
    """Aggregate the maximum non-task language probability per row."""
    if USE_CACHE and MAX_OTHER_LANGDIST_CACHE_PATH.exists():
        print(f"Loading max-other aggregation cache: {MAX_OTHER_LANGDIST_CACHE_PATH.relative_to(REPO_ROOT)}")
        return pd.read_parquet(MAX_OTHER_LANGDIST_CACHE_PATH)

    records = []
    translation_runs = _load_translation_runs_for_language_agg()
    for row in tqdm(
        translation_runs.itertuples(index=False),
        total=len(translation_runs),
        desc="Finding max other language for translation",
        unit="run",
    ):
        if not isinstance(row.exp_path, str):
            continue
        artifact_path = Path(row.exp_path) / f"lang_probs_{row.revision}.pt"
        if not artifact_path.exists():
            continue
        artifact = load_lang_probs_artifact(artifact_path)
        payload = artifact.get("methods", {}).get("decoding", {})
        if "probs" not in payload:
            continue
        langs = list(payload.get("langs") or [])
        probs = payload["probs"].to(dtype=torch.float32).numpy()
        prompt_langs = list(artifact.get("prompt_langs") or [None] * probs.shape[0])
        layer_indices = list(artifact.get("layer_indices") or list(range(probs.shape[1])))
        requested_tgt_lang = row.requested_tgt_lang
        records.extend(_accumulate_max_other_sums(
            probs=probs,
            langs=langs,
            prompt_langs=prompt_langs,
            layer_indices=layer_indices,
            windows=COMPARISON_WINDOWS,
            base_fields={"method_label": "target-start", "display_model_name": row.display_model_name},
            other_lang_fn=lambda lang, prompt_lang, requested_tgt_lang=requested_tgt_lang: (
                lang not in {"en", prompt_lang, requested_tgt_lang}
            ),
            skip_prompt_fn=lambda prompt_lang, requested_tgt_lang=requested_tgt_lang: (
                prompt_lang == "en" or requested_tgt_lang == "en"
            ),
        ))

    include_runs = manifest[
        manifest["data_source"].eq(INCLUDE_DATA_SOURCE)
        & manifest["method_label"].isin(["raw-rtopp", "tuned-rtopp", "repr"])
        & manifest["lang_probs_artifact_exists"]
    ].copy()
    for row in tqdm(
        include_runs.itertuples(index=False),
        total=len(include_runs),
        desc="Finding max other language for INCLUDE",
        unit="run",
    ):
        artifact = load_lang_probs_artifact(row.lang_probs_artifact_path)
        payload = artifact.get("methods", {}).get(_artifact_method_for_label(row.method_label), {})
        if "probs" not in payload:
            continue
        langs = list(payload.get("langs") or [])
        probs = payload["probs"].to(dtype=torch.float32).numpy()
        prompt_langs = list(artifact.get("prompt_langs") or [None] * probs.shape[0])
        layer_indices = list(artifact.get("layer_indices") or list(range(probs.shape[1])))
        records.extend(_accumulate_max_other_sums(
            probs=probs,
            langs=langs,
            prompt_langs=prompt_langs,
            layer_indices=layer_indices,
            windows=COMPARISON_WINDOWS,
            base_fields={"method_label": row.method_label, "display_model_name": row.display_model_name},
            other_lang_fn=lambda lang, prompt_lang: lang not in {"en", prompt_lang},
            skip_prompt_fn=lambda prompt_lang: prompt_lang == "en",
        ))

    agg = pd.DataFrame.from_records(records)
    if not agg.empty:
        agg = agg.groupby(
            ["method_label", "display_model_name", "layer_window", "layer_window_slug", "prob_category", "lang"],
            dropna=False,
        ).agg(
            sum_prob=("sum_prob", "sum"),
            n_values=("n_values", "sum"),
            sum_sq_prob=("sum_sq_prob", "sum"),
        ).reset_index()
        agg["lang_mean_prob"] = agg["sum_prob"] / agg["n_values"].clip(lower=1)
    agg.to_parquet(MAX_OTHER_LANGDIST_CACHE_PATH, index=False)
    print(f"Wrote max-other aggregation cache: {MAX_OTHER_LANGDIST_CACHE_PATH.relative_to(REPO_ROOT)}")
    return agg


max_other_agg = build_max_other_language_aggregates()
max_other_summary = summarize_language_aggregates(max_other_agg)
print(f"Max-other summary rows: {len(max_other_summary):,}")


In [ ]:
# Assemble the average and maximum probability summaries used by Figure 4.
MAX_TASK_LANGDIST_CACHE_PATH = CACHE_DIR / "synthetic_translation_late_layer_max_task_langdist.parquet"


def build_max_task_relevant_language_aggregates():
    """Aggregate the maximum source/target language probability per row."""
    if USE_CACHE and MAX_TASK_LANGDIST_CACHE_PATH.exists():
        print(f"Loading max-task-relevant cache: {MAX_TASK_LANGDIST_CACHE_PATH.relative_to(REPO_ROOT)}")
        return pd.read_parquet(MAX_TASK_LANGDIST_CACHE_PATH)

    records = []
    translation_runs = _load_translation_runs_for_language_agg()
    for row in tqdm(
        translation_runs.itertuples(index=False),
        total=len(translation_runs),
        desc="Finding max task-relevant language for translation",
        unit="run",
    ):
        if not isinstance(row.exp_path, str):
            continue
        artifact_path = Path(row.exp_path) / f"lang_probs_{row.revision}.pt"
        if not artifact_path.exists():
            continue
        artifact = load_lang_probs_artifact(artifact_path)
        payload = artifact.get("methods", {}).get("decoding", {})
        if "probs" not in payload:
            continue
        langs = list(payload.get("langs") or [])
        probs = payload["probs"].to(dtype=torch.float32).numpy()
        prompt_langs = list(artifact.get("prompt_langs") or [None] * probs.shape[0])
        layer_indices = list(artifact.get("layer_indices") or list(range(probs.shape[1])))
        requested_tgt_lang = row.requested_tgt_lang
        records.extend(_accumulate_max_other_sums(
            probs=probs,
            langs=langs,
            prompt_langs=prompt_langs,
            layer_indices=layer_indices,
            windows=COMPARISON_WINDOWS,
            base_fields={"method_label": "target-start", "display_model_name": row.display_model_name},
            other_lang_fn=lambda lang, prompt_lang, requested_tgt_lang=requested_tgt_lang: (
                lang in {prompt_lang, requested_tgt_lang}
            ),
            skip_prompt_fn=lambda prompt_lang, requested_tgt_lang=requested_tgt_lang: (
                prompt_lang == "en" or requested_tgt_lang == "en"
            ),
        ))

    agg = pd.DataFrame.from_records(records)
    if not agg.empty:
        agg["prob_category"] = "task_relevant_max"
        agg["lang"] = "__max_task_relevant__"
        agg = agg.groupby(
            ["method_label", "display_model_name", "layer_window", "layer_window_slug", "prob_category", "lang"],
            dropna=False,
        ).agg(
            sum_prob=("sum_prob", "sum"),
            n_values=("n_values", "sum"),
            sum_sq_prob=("sum_sq_prob", "sum"),
        ).reset_index()
        agg["lang_mean_prob"] = agg["sum_prob"] / agg["n_values"].clip(lower=1)
    agg.to_parquet(MAX_TASK_LANGDIST_CACHE_PATH, index=False)
    print(f"Wrote max-task-relevant cache: {MAX_TASK_LANGDIST_CACHE_PATH.relative_to(REPO_ROOT)}")
    return agg


max_task_relevant_agg = build_max_task_relevant_language_aggregates()
translation_max_task_summary = summarize_language_aggregates(max_task_relevant_agg)

avg_task_summary = micro_summary[micro_summary["prob_category"].eq("task_relevant")].copy()
avg_task_summary["prob_category"] = "task_relevant_avg"
include_max_task_summary = micro_summary[
    micro_summary["prob_category"].eq("task_relevant")
    & ~micro_summary["method_label"].eq("target-start")
].copy()
include_max_task_summary["prob_category"] = "task_relevant_max"
english_summary = micro_summary[micro_summary["prob_category"].eq("english")].copy()
avg_other_summary = micro_summary[micro_summary["prob_category"].eq("other")].copy()
avg_other_summary["prob_category"] = "other_avg"
combined_max_other_summary = max_other_summary.copy()
combined_max_other_summary["prob_category"] = "other_max"

five_category_summary = pd.concat(
    [
        avg_task_summary,
        translation_max_task_summary,
        include_max_task_summary,
        english_summary,
        avg_other_summary,
        combined_max_other_summary,
    ],
    ignore_index=True,
)
FIVE_CATEGORY_COLORS = {
    "task_relevant_avg": "#2f7ed8",
    "task_relevant_max": "#56B4E9",
    "english": "#d95f02",
    "other_avg": "#999999",
    "other_max": "#3f3f3f",
}



In [ ]:
# Paper Figure 4: average probability bars with per-row maximum markers.
AVG_MAX_BAR_CATEGORY_MAP = {
    "task_relevant_avg": "task_relevant",
    "english": "english",
    "other_avg": "other",
}
AVG_MAX_BAR_LABELS = {
    "task_relevant": "Avg. P(L=task-relevant)",
    "english": "P(L=English)",
    "other": "Avg. P(L=other)",
}
AVG_MAX_BAR_COLORS = {
    "task_relevant": FIVE_CATEGORY_COLORS["task_relevant_avg"],
    "english": FIVE_CATEGORY_COLORS["english"],
    "other": FIVE_CATEGORY_COLORS["other_avg"],
}

avg_max_bar_summary = five_category_summary[
    five_category_summary["prob_category"].isin(AVG_MAX_BAR_CATEGORY_MAP)
].copy()
avg_max_bar_summary["prob_category"] = avg_max_bar_summary["prob_category"].map(AVG_MAX_BAR_CATEGORY_MAP)
avg_max_marker_summary = five_category_summary[
    five_category_summary["prob_category"].isin(["task_relevant_max", "other_max"])
].copy()

avg_max_bar_plot_df = add_late_window_comparison_labels(avg_max_bar_summary)
avg_max_marker_plot_df = add_late_window_comparison_labels(avg_max_marker_summary)
avg_max_scale_df = pd.concat([avg_max_bar_plot_df, avg_max_marker_plot_df], ignore_index=True)
avg_max_models = [
    model
    for model in TABLE1_REVERSE_MODEL_ORDER
    if model in set(avg_max_bar_summary["display_model_name"].dropna())
    and model not in COMPARISON_ROW_TEMP_EXCLUDE_MODELS
]

fig = plot_story_category_bar_grid(
    avg_max_bar_plot_df,
    row_col="comparison_layer_row",
    row_order=[COMPARISON_ROW_LABEL_MAP[row] for row in COMPARISON_ROW_ORDER],
    col_col="comparison_task_panel",
    col_order=COMPARISON_TASK_PANEL_ORDER,
    models=avg_max_models,
    category_order=BAR_ORDER,
    category_labels=AVG_MAX_BAR_LABELS,
    category_colors=AVG_MAX_BAR_COLORS,
    probability_label="Average Language Probability",
    shared_ylabel="Avg. Lang Prob across Prompts and Layers",
    show_row_label_in_ylabel=True,
    title=None,
    subtitle=None,
    unavailable_label="average/max overlay row plot",
    figsize=(16.5, 6.4),
    legend_y=0.010,
    bottom=0.285,
    top=0.870,
    left=0.110,
    right=0.990,
    shared_ylabel_x=0.020,
    sharey=False,
    model_label_map=COMPARISON_MODEL_LABEL_MAP,
)
if fig is not None:
    decorate_late_window_comparison_by_window_rows(fig, avg_max_scale_df)
    panel_count = len(COMPARISON_ROW_ORDER) * len(COMPARISON_TASK_PANEL_ORDER)
    panel_axes = np.asarray(fig.axes[:panel_count], dtype=object).reshape(
        len(COMPARISON_ROW_ORDER), len(COMPARISON_TASK_PANEL_ORDER)
    )
    category_offsets = {category: (idx - 1) * 0.22 for idx, category in enumerate(BAR_ORDER)}
    marker_specs = [
        ("task_relevant", "task_relevant_max"),
        ("other", "other_max"),
    ]
    row_labels = [COMPARISON_ROW_LABEL_MAP[row] for row in COMPARISON_ROW_ORDER]
    for row_idx, row_label in enumerate(row_labels):
        for col_idx, panel_label in enumerate(COMPARISON_TASK_PANEL_ORDER):
            ax = panel_axes[row_idx, col_idx]
            for model_idx, model in enumerate(avg_max_models):
                for avg_category, max_category in marker_specs:
                    avg_rows = avg_max_bar_plot_df[
                        avg_max_bar_plot_df["comparison_layer_row"].eq(row_label)
                        & avg_max_bar_plot_df["comparison_task_panel"].eq(panel_label)
                        & avg_max_bar_plot_df["display_model_name"].eq(model)
                        & avg_max_bar_plot_df["prob_category"].eq(avg_category)
                    ]
                    max_rows = avg_max_marker_plot_df[
                        avg_max_marker_plot_df["comparison_layer_row"].eq(row_label)
                        & avg_max_marker_plot_df["comparison_task_panel"].eq(panel_label)
                        & avg_max_marker_plot_df["display_model_name"].eq(model)
                        & avg_max_marker_plot_df["prob_category"].eq(max_category)
                    ]
                    if avg_rows.empty or max_rows.empty:
                        continue
                    avg_prob = float(avg_rows["mean_prob"].mean())
                    max_prob = float(max_rows["mean_prob"].mean())
                    max_se = float(max_rows["stderr_prob"].mean()) if "stderr_prob" in max_rows else np.nan
                    x_pos = model_idx + category_offsets[avg_category]
                    color = AVG_MAX_BAR_COLORS[avg_category]
                    ax.vlines(x_pos, avg_prob, max_prob, color=color, linewidth=1.5, zorder=4)
                    ax.errorbar(
                        x_pos,
                        max_prob,
                        yerr=max_se if np.isfinite(max_se) else None,
                        fmt="D",
                        markersize=5.5,
                        markerfacecolor="white",
                        markeredgecolor=color,
                        markeredgewidth=1.5,
                        ecolor=color,
                        elinewidth=1.2,
                        capsize=3.0,
                        zorder=6,
                        label=None,
                    )
            ax.set_xlim(-0.46, len(avg_max_models) - 1 + 0.46)
    for row_ax in panel_axes[:, 0]:
        row_ax.yaxis.set_label_coords(-0.16, 0.5)
    if getattr(fig, "_supylabel", None) is not None:
        panel_y0 = min(ax.get_position().y0 for ax in panel_axes.ravel())
        panel_y1 = max(ax.get_position().y1 for ax in panel_axes.ravel())
        fig._supylabel.set_y((panel_y0 + panel_y1) / 2.0)

    handles, labels = panel_axes[0, 0].get_legend_handles_labels()
    handles.append(plt.Line2D(
        [], [], linestyle="none", marker="D", markersize=6.0,
        markerfacecolor="white", markeredgecolor="black", markeredgewidth=1.5,
    ))
    labels.append("Avg. Max. P(L)")
    for old_legend in list(fig.legends):
        old_legend.remove()
    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.010),
        ncol=len(handles),
        frameon=False,
        fontsize=18,
        handlelength=1.35,
        handletextpad=0.55,
    )
    # Shift labels, not ticks, by a small constant amount.
    for label_ax in panel_axes[-1, :]:
        for tick_label in label_ax.get_xticklabels():
            tick_label.set_transform(
                tick_label.get_transform()
                + plt.matplotlib.transforms.ScaledTranslation(
                    14.0 / 72.0, 0.0, fig.dpi_scale_trans
                )
            )
    # Align the top of the shared y-label with the controlled-task heading.
    fig.canvas.draw()
    if getattr(fig, "_supylabel", None) is not None:
        controlled_heading = next(
            (text for text in fig.texts if text.get_text() == "Controlled: translation"),
            None,
        )
        if controlled_heading is not None:
            renderer = fig.canvas.get_renderer()
            ylabel_top = fig._supylabel.get_window_extent(renderer=renderer).y1
            heading_top = controlled_heading.get_window_extent(renderer=renderer).y1
            current_y = fig._supylabel.get_position()[1]
            fig._supylabel.set_y(current_y + (heading_top - ylabel_top) / fig.bbox.height)
    save_matplotlib_figure_bundle(
        fig,
        PAPER_MAIN_FIG_DIR / "04_language_probs_late_layer_windows",
    )
    plt.show()
